In [7]:
"""
SP-Broyden: данные и графики для тезисов конференции МФТИ.
Результат: fig_tezisy.png
"""

import numpy as np
from numpy.linalg import norm, solve, cond
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")


# ═══════════════════════════════════════════
#  Солвер SP-Broyden
# ═══════════════════════════════════════════

def sp_broyden_solve(F, x0, p_max=0, reset=False,
                     cond_thresh=1e3, maxiter=500, tol=1e-13):
    """
    Параметры:
      p_max=0,  reset=False  →  классический Бройден
      p_max>0,  reset=False  →  SP-Broyden (секанто-сохраняющий)
      p_max=m,  reset=True   →  Андерсон (мультисекущая пересборка)
    Возвращает: list of (iteration, f_evals, ||F||)
    """
    n = len(x0)
    x = x0.copy().astype(float)
    Fx = F(x)
    f_evals = 1
    hist = [(0, 1, float(norm(Fx)))]

    B = np.eye(n)
    S_hist, Y_hist = [], []

    for k in range(maxiter):
        if norm(Fx) < tol:
            break

        try:
            d = solve(B, -Fx)
        except np.linalg.LinAlgError:
            break
        if not np.all(np.isfinite(d)):
            break

        x_new = x + d
        Fx_new = F(x_new)
        f_evals += 1
        if not np.all(np.isfinite(Fx_new)):
            break

        y = Fx_new - Fx
        s = d
        S_hist.append(s.copy())
        Y_hist.append(y.copy())

        if reset:
            # ── Ветка Андерсона: пересборка B = I + (Y-S)(S^T S)^{-1} S^T ──
            m = min(p_max, len(S_hist))
            B = np.eye(n)
            if m > 0:
                Sm = np.column_stack(S_hist[-m:])
                Ym = np.column_stack(Y_hist[-m:])
                G = Sm.T @ Sm
                while m > 0 and cond(G) > cond_thresh:
                    m -= 1
                    if m == 0:
                        break
                    Sm = np.column_stack(S_hist[-m:])
                    Ym = np.column_stack(Y_hist[-m:])
                    G = Sm.T @ Sm
                if m > 0:
                    try:
                        B = np.eye(n) + (Ym - Sm) @ solve(G, Sm.T)
                    except np.linalg.LinAlgError:
                        B = np.eye(n)
        else:
            # ── Ветка Бройдена: ранг-1 с выбором v ──
            Bs = B @ s
            p = 0
            if p_max > 0 and len(S_hist) >= 2:
                for p_try in range(1, min(p_max, len(S_hist) - 1) + 1):
                    cols = [S_hist[-1 - j] for j in range(p_try + 1)]
                    Sp = np.column_stack(cols)
                    G = Sp.T @ Sp
                    if cond(G) < cond_thresh:
                        p = p_try
                    else:
                        print(k, cond(G))
                        break

            if p == 0:
                v = s  # классический Бройден
            else:
                cols = [S_hist[-1 - j] for j in range(p + 1)]
                Sp = np.column_stack(cols)
                G = Sp.T @ Sp
                e1 = np.zeros(p + 1)
                e1[0] = 1.0
                try:
                    v = Sp @ solve(G, e1)
                except np.linalg.LinAlgError:
                    v = s

            denom = v @ s
            if abs(denom) < 1e-12:
                v = s
                denom = s @ s
            # if abs(denom) < 1e-12:
            #     break

            B = B + np.outer(y - Bs, v) / denom

        x = x_new
        Fx = Fx_new
        hist.append((k + 1, f_evals, float(norm(Fx))))

        max_hist = max(p_max + 5, 15)
        if len(S_hist) > max_hist:
            S_hist.pop(0)
            Y_hist.pop(0)

    return hist


# ═══════════════════════════════════════════
#  Тестовые задачи
# ═══════════════════════════════════════════

def discrete_bvp(x):
    n = len(x)
    h = 1.0 / (n + 1)
    r = np.zeros(n)
    for i in range(n):
        ti = (i + 1) * h
        xm = x[i - 1] if i > 0 else 0.0
        xp = x[i + 1] if i < n - 1 else 0.0
        r[i] = 2 * x[i] - xm - xp + h**2 * (x[i] + ti + 1)**3 / 2
    return r

def discrete_bvp_x0(n):
    h = 1.0 / (n + 1)
    return np.array([i * h * (i * h - 1) for i in range(1, n + 1)]) * 0.1

def broyden_banded(x):
    n = len(x)
    r = np.zeros(n)
    for i in range(n):
        ji = [j for j in range(max(0, i - 5), min(n, i + 2)) if j != i]
        r[i] = x[i] * (2 + 5 * x[i]**2) + 1 - sum(x[j] * (1 + x[j]) for j in ji)
    return r

def broyden_banded_x0(n):
    return -np.ones(n) * 0.1


# ═══════════════════════════════════════════
#  Методы для сравнения
# ═══════════════════════════════════════════

METHODS = [
    # (label, p_max, reset, color, linestyle, linewidth)
    ("Бройден",          0,  False, "#888888", (0, (4, 3)),  1.3),
    ("Андерсон (m=10)",  10, True,  "#2060B0", (0, (5, 2)),  1.5),
    ("SP-Broyden (p≤10)", 10, False, "#D03030", "-",         2.3),
]

PROBLEMS = [
    ("Discrete BVP, n = 100",   discrete_bvp,    discrete_bvp_x0(100), 1e-15),
    ("Broyden Banded, n = 100", broyden_banded,  broyden_banded_x0(100), 1e-12),
]


# ═══════════════════════════════════════════
#  Запуск и печать результатов
# ═══════════════════════════════════════════

print("=" * 70)
print("  SP-Broyden: данные для тезисов  |  B_0 = I  |  tol = 1e-10")
print("=" * 70)

all_data = {}

for prob_name, F, x0, tol in PROBLEMS:
    print(f"\n  {prob_name}")
    prob_data = {}
    for label, pm, rst, *_ in METHODS:
        h = sp_broyden_solve(F, x0.copy(), p_max=pm, reset=rst, tol=tol)
        final_res = h[-1][2]
        n_iter = h[-1][0]
        conv = final_res < 1e-10
        print(f"    {label:<22s}  iter={n_iter:<4d}  ||F||={final_res:.2e}")
        prob_data[label] = h
    all_data[prob_name] = prob_data


# ═══════════════════════════════════════════
#  Построение графиков
# ═══════════════════════════════════════════

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(5.2, 2.4))

for ax, (prob_name, F, x0, tol) in zip([ax1, ax2], PROBLEMS):
    data = all_data[prob_name]

    for label, pm, rst, color, ls, lw in METHODS:
        h = data[label]
        iters = [t[0] for t in h]
        resid = [t[2] for t in h]

        # Фильтруем inf/nan
        xs, ys = [], []
        for i, r in zip(iters, resid):
            if np.isfinite(r) and r > 0:
                xs.append(i)
                ys.append(r)
            else:
                break

        if len(xs) < 2:
            continue

        ax.semilogy(xs, ys, color=color, linestyle=ls,
                    linewidth=lw, label=f"{label}", alpha=0.9)

    ax.axhline(1e-10, color="#00BBCC", lw=0.5, ls=":", alpha=0.4)

    ax.set_title(prob_name, fontsize=8, pad=4)
    ax.set_xlabel("итерация k", fontsize=7)
    ax.set_ylabel("‖F(x)‖", fontsize=7)
    ax.tick_params(labelsize=6)
    ax.grid(True, alpha=0.2)

ax2.set_ylim(bottom=1e-14, top=1e8)

ax2.legend(fontsize=5.5, loc="upper right", framealpha=0.8,
           borderpad=0.3, handlelength=1.5, labelspacing=0.2)

fig.tight_layout(pad=0.5)
fig.savefig("fig_tezisy.png", dpi=300, bbox_inches="tight", pad_inches=0.05)
print(f"\nГрафик сохранен: fig_tezisy.png")
plt.close()

  SP-Broyden: данные для тезисов  |  B_0 = I  |  tol = 1e-10

  Discrete BVP, n = 100
    Бройден                 iter=371   ||F||=9.85e-16
    Андерсон (m=10)         iter=500   ||F||=1.26e-08
1 14833.416397577777
2 15498.735311196979
3 38113.5764166505
4 9539.048518845939
5 92858.98250959592
6 96872.94176555106
7 631358.5970917529
8 2030704.0187482107
9 1601.589867684399
10 10868.384119566272
11 20325.30435628772
12 3500.7040015868856
13 17136.23017816046
14 36197.12918340071
15 272327.79248052667
16 1225.769901087592
17 3495.316292932632
18 7292.334074895924
19 1454.1544917096567
20 1754.3941471502737
21 2207.4742895529953
22 4327.589602911613
23 4503.077324501055
24 41975.88134202823
25 71482.68608615115
26 2086.2072015749823
27 1289.9354936976995
28 690534.3272370296
29 231288.7303256549
30 151984.47227703183
31 351049.69631697366
32 1919.6626908049716
33 2743.4636756875443
34 28392.320643209245
35 1891.0546705643183
36 12889.235799649154
37 18892.70546645365
38 120918.48295661174

**Вход:** $F\colon \mathbb{R}^n \to \mathbb{R}^n$, $x_0 \in \mathbb{R}^n$, $p_{\max} \geq 0$, $\texttt{reset} \in \{\texttt{true}, \texttt{false}\}$, $\tau > 0$, $\varepsilon > 0$.

**Инициализация:** $B_0 = I_n$, $\mathcal{S} = \varnothing$, $\mathcal{Y} = \varnothing$.

---

**Цикл** (пока $\|F(x_k)\| \geq \varepsilon$):

1. Решить $B_k\, d_k = -F(x_k)$.

2. $x_{k+1} \leftarrow x_k + d_k$, $\;s_k \leftarrow d_k$, $\;y_k \leftarrow F(x_{k+1}) - F(x_k)$.

3. Добавить $s_k$ в $\mathcal{S}$, $\;y_k$ в $\mathcal{Y}$.

4. **Обновление $B_k$:**

   **Если $\texttt{reset} = \texttt{true}$:**

   > Пересборка из $I$ (ветка Андерсона):
   >
   > $m \leftarrow \max\{m' \leq p_{\max} : \operatorname{cond}(S_{m'}^\top S_{m'}) < \tau\}$
   >
   > $B_{k+1} \leftarrow I + (Y_m - S_m)(S_m^\top S_m)^{-1} S_m^\top$

   **Если $\texttt{reset} = \texttt{false}$:**

   > Ранг-1 обновление:
   >
   > $p \leftarrow \max\{p' \leq p_{\max} : \operatorname{cond}(S_{p'}^\top S_{p'}) < \tau\}$
   >
   > **Если $p = 0$:** $\;v_k \leftarrow s_k$
   >
   > **Если $p > 0$:** $\;v_k \leftarrow S_p\,(S_p^\top S_p)^{-1}\,e_1$ $\quad$ ⟵ ** отличие от классического Бройдена**
   >
   > $B_{k+1} \leftarrow B_k + \dfrac{(y_k - B_k s_k)\,v_k^\top}{v_k^\top s_k}$

5. $k \leftarrow k + 1$.

---

## Частные случаи

| Метод | $p_{\max}$ | reset | Что происходит |
|:------|:----------:|:-----:|:---------------|
| Классический Бройден | $0$ | false | $v_k = s_k$ всегда, ранг-1 обновление |
| **SP-Broyden** | $p > 0$ | false | $v_k = S_p(S_p^\top S_p)^{-1}e_1$, ранг-1 обновление |
| Мультисекущий Бройден | $k$ | false | SP-Broyden с полной историей |
| Андерсон | $m$ | true | Пересборка $B$ из $I$ на каждом шаге |

---

## Что именно отличается

### SP-Broyden vs классический Бройден

Отличие ровно в одной строке — **выбор вектора $v_k$**:

```
Бройден:     v_k = s_k
SP-Broyden:  v_k = S_p (S_p^T S_p)^{-1} e_1    (при p > 0)
```

Все остальное (линейная система, шаг, ранг-1 формула обновления) — идентично.

**Следствие этого отличия:**

| | Бройден ($v_k = s_k$) | SP-Broyden ($v_k = v_k^*$) |
|:--|:--|:--|
| Текущее секущее $B_{k+1}s_k = y_k$ | ✓ выполняется | ✓ выполняется |
| Прежнее секущее $B_{k+1}s_j = y_j$ | ✗ нарушается на $O(\|x_k - x^*\|)$ | ✓ сохраняется точно |
| Минимальность $\|B_{k+1} - B_k\|_F$ | минимум **без** ограничений на старые секущие | минимум **с** ограничениями сохранения |
| Затраты на шаг | $O(n^2)$ | $O(n^2 + np^2)$ |

### SP-Broyden vs Андерсон

Отличие — в **механизме обновления**:

```
Андерсон:    B_{k+1} = I + (Y_m - S_m)(S_m^T S_m)^{-1} S_m^T    (пересборка с нуля)
SP-Broyden:  B_{k+1} = B_k + (y_k - B_k s_k) v_k^T / (v_k^T s_k)  (ранг-1 к текущему B_k)
```

| | Андерсон (reset = true) | SP-Broyden (reset = false) |
|:--|:--|:--|
| Информация за пределами окна $m$ | **стирается** (B пересобирается из I) | **накапливается** в $B_k$ |
| Ранг обновления | до $m$ (полная пересборка) | 1 (инкрементальное) |
| Устойчивость | высокая (I — хороший якорь) | зависит от качества $B_k$ |
| Выгоден когда | якобиан быстро меняется | якобиан меняется медленно |

### SP-Broyden vs мультисекущий Бройден

Мультисекущий Бройден — это SP-Broyden с $p = k$ (сохраняем **все** секущие условия):

```
Мультисекущий:  p = k   (полная история, грамиан растет)
SP-Broyden:     p ≤ p_max, адаптивный  (окно фиксировано)
```

| | Мультисекущий ($p = k$) | SP-Broyden ($p \leq p_{\max}$) |
|:--|:--|:--|
| Память | $O(nk)$ — растет с итерациями | $O(np_{\max})$ — фиксирована |
| Грамиан | вырождается при $k > n$ | контролируется порогом $\tau$ |

---